# 2. Alignment, dimensional collapse, and loss ablations

**Question:** Does a locality advantage survive objectives that discourage low-rank representations, or is dimensional collapse central to the observed behavior?

Two encoders learn separately from their own temporal observation pairs. No cross-modal targets, state labels, or shared optimizer are used. Compare the original objective with stronger variance, covariance, and covariance-identity penalties. High coordinate variance is not enough: several coordinates can still encode the same direction.

Start with a small development plan. Use new world IDs for confirmation after settings are frozen. Retain all runs, including collapse flags and undefined metrics. A favorable result is not required.

**Storage:** results remain in memory unless `SAVE_RESULTS` is explicitly enabled with a destination. Editor autosave may separately store displayed cell output in the notebook; clear outputs or disable that feature if needed.

In [ ]:
from pathlib import Path
import sys
import subprocess
from zipfile import ZipFile

REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_REF = 'main'  # For a frozen run, choose a commit before starting a fresh runtime.
SOURCE_ARCHIVE = None  # Optional uploaded source ZIP; not needed in Colab with GitHub access.
INSTALL_DEPENDENCIES = False  # Enable only if the runtime lacks compatible dependencies.
cloud = next((p for p in (Path('/content'), Path('/kaggle/working')) if p.exists()), None)
if SOURCE_ARCHIVE is not None:
    if cloud is None:
        raise RuntimeError('Use the local checkout here; source extraction is for cloud runtimes.')
    destination = cloud / 'dynalign_uploaded_source'
    with ZipFile(SOURCE_ARCHIVE) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive member')
        archive.extractall(destination)
    candidates = [destination]
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src/dynalign/revised.py').exists()), None)
if ROOT is None and SOURCE_ARCHIVE is None and cloud is not None:
    destination = cloud / 'dynalign_github'
    if not destination.exists():
        subprocess.check_call(['git', 'clone', '--quiet', REPO_URL, str(destination)])
        subprocess.check_call(['git', '-C', str(destination), 'fetch', '--quiet', 'origin', REPO_REF])
        subprocess.check_call(['git', '-C', str(destination), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'])
    ROOT = destination
if ROOT is None or not (ROOT / 'src/dynalign/revised.py').exists():
    raise FileNotFoundError('Open the current checkout locally, or use a fresh Colab/Kaggle runtime with Internet enabled.')
# Existing checkouts are deliberately reused, not reset or silently updated.
# Restart in a fresh runtime to switch REPO_REF or load a newer GitHub revision.
if 'dynalign' in sys.modules and Path(sys.modules['dynalign'].__file__).resolve().parents[2] != ROOT.resolve():
    raise RuntimeError('Another source version is already imported; restart the runtime before switching.')
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True)
print('Source revision:', revision.stdout.strip() if revision.returncode == 0 else 'uploaded source bundle')
print('Source setup only: result saving remains disabled by default.')

from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynalign.revised import FitSettings, EvalSettings, SyntheticSpec, run_synthetic, preview_plan, paired_contrasts, bundle_zip

## Experiments to try

| Stage | Change in the configuration cell | What it tests |
|---|---|---|
| Execution check | One world, 256 pairs, 40–80 updates | Code paths only; no scientific inference |
| Loss diagnostic | 2,048 pairs, width 64, four objectives; checkpoints 800 and 3,200; development worlds | Whether variance/covariance tradeoffs explain dimensional collapse |
| Confirmation | 25 **new** world IDs, two repeats; baseline plus one previously frozen candidate | Whether the contrast generalizes beyond development worlds |
| Negative control | Add `iid` to conditions for a small diagnostic subset | Behavior without useful temporal state dependence; not entropy matched |

Weights are listed as invariance / variance / off-diagonal covariance. The covariance-identity variant instead uses invariance plus a full covariance-to-identity penalty. It is an alternative training objective, **not post-hoc whitening or a guaranteed rank constraint**.

Apply the same objective to both conditions. Choose any candidate using training/validation nondegeneracy and stability on development worlds, never the largest favorable test contrast. Keep those worlds out of confirmation. Do not remove collapsed runs, and do not inflate rank with measurement noise.

In [ ]:
# Edit this cell, inspect the plan, then enable RUN_TRAINING.
RUN_TRAINING = False
PHASE = 'development'
WORLD_IDS = [100]  # Confirmation example: list(range(1000, 1025)); never reuse development worlds.
REPEAT_IDS = [0]   # Expanded runs: [0, 1]
CONDITIONS = ['local', 'scrambled']
DATA_BUDGETS = [256]  # Loss diagnostic: [2048]
MASTER_SEED = 20261108
N_STATES, SENSOR_WIDTH = 32, 12
ARCHITECTURE, WIDTH, EMBEDDING_DIM = 'mlp', 64, 4
NOISE = 0.05
DEVICE, CPU_THREADS = 'auto', 2
SAVE_RESULTS = False
OUTPUT_DIR = None  # Explicit cloud/durable path only if SAVE_RESULTS=True.

FIT = FitSettings(steps=80, checkpoints=(40, 80), batch_size=128, lr=1e-3,
                  weight_decay=1e-4, invariance_weight=25, variance_weight=25,
                  covariance_weight=1, identity_weight=25, variance_target=1,
                  epsilon=1e-4, clip_grad=5, log_every=40, checkpoint_every=100)
EVAL = EvalSettings(calibration_repeats=8, test_repeats=16, ridge=1e-3,
                    collapse_variance=1e-6, collapse_rank_fraction=0.5, batch_size=512)
REGIMES = {
    'baseline': dict(objective='vicreg', invariance_weight=25, variance_weight=25, covariance_weight=1),
    'strong_variance': dict(objective='vicreg', invariance_weight=25, variance_weight=100, covariance_weight=1),
    'strong_covariance': dict(objective='vicreg', invariance_weight=25, variance_weight=25, covariance_weight=10),
    'identity_candidate': dict(objective='cov_identity', invariance_weight=25, identity_weight=25),
}
ACTIVE_REGIMES = ['baseline', 'strong_variance', 'strong_covariance', 'identity_candidate']
if SAVE_RESULTS and OUTPUT_DIR is None:
    raise ValueError('Set an explicit OUTPUT_DIR before saving.')
specs = [SyntheticSpec(experiment='alignment_collapse', phase=PHASE, world=w, repeat=r,
                      master_seed=MASTER_SEED, condition=c, n_states=N_STATES, n_pairs=n,
                      sensor_width=SENSOR_WIDTH, noise=NOISE, architecture=ARCHITECTURE,
                      width=WIDTH, dim=EMBEDDING_DIM, regime=regime,
                      fit=replace(FIT, **REGIMES[regime]), evaluation=EVAL)
         for w in WORLD_IDS for r in REPEAT_IDS for n in DATA_BUDGETS
         for regime in ACTIVE_REGIMES for c in CONDITIONS]
display(preview_plan(specs))
print(f'{len(specs)} paired-encoder runs = {2 * len(specs)} encoder fits. Training enabled: {RUN_TRAINING}')

## Execute only the plan you inspected

Set `RUN_TRAINING=True` above and rerun the configuration cell to train. **Edit loss weights in `REGIMES`: these override the corresponding defaults in `FIT`.** Other optimizer settings come from `FIT`. The displayed plan shows the effective merged settings that will actually run.

The runner evaluates step 0 plus the specified checkpoints, uses separate modality-specific random streams, and does not tune from test scores. All settings are part of the recorded plan and run identity.

With saving disabled there are no checkpoints, CSVs, archives on disk, or automatic downloads; progress is lost if the runtime ends. If saving is enabled, checkpoints and incremental result archives go only to your explicit destination. Only resume checkpoints you trust.

In [ ]:
bundle = None
if RUN_TRAINING:
    bundle = run_synthetic(specs, device=DEVICE, threads=CPU_THREADS,
                           output_dir=OUTPUT_DIR if SAVE_RESULTS else None)
    results = bundle.table
    display(results[['regime', 'condition', 'world', 'repeat', 'step', 'cka',
                     'effective_rank_a', 'effective_rank_b', 'any_collapse',
                     'estimated_signal_fraction_a', 'estimated_signal_fraction_b']])
else:
    print('Plan only. Set RUN_TRAINING=True in the configuration cell to execute it.')

## Read geometry, rank, and information together

Primary comparison: local minus scrambled at a fixed regime, budget, and update count. Each repeat is paired first, then repeats are averaged within world. An incomplete pair is reported and prevents that world from silently contributing a survivor-only average. With fewer than five valid worlds, no bootstrap interval is returned; 25 worlds is still a design choice, not a power guarantee.

Inspect all covariance eigenvalues, low-rank flags, and estimated state-mean versus within-state noise variation. State-mean variance uses repeated measurements and a finite-repeat noise correction; negative estimates are possible from sampling and are retained. The displayed signal fraction is clipped only for interpretation. High rank with mostly within-state noise is not useful recovery.

Do not select a regime from these test contrasts. Compare nondegeneracy and stability on separate development/validation data before freezing the confirmation plan.

In [ ]:
if bundle is not None:
    contrasts = paired_contrasts(results, metric='cka', group_by=['regime', 'n_pairs', 'step'])
    display(contrasts)
    last = results[results.step == FIT.steps]
    display(last[['regime', 'condition', 'world', 'repeat', 'eigenvalues_a', 'eigenvalues_b',
                  'state_mean_variance_corrected_a', 'within_state_noise_variance_a',
                  'state_mean_variance_corrected_b', 'within_state_noise_variance_b', 'any_collapse']])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for (regime, condition), block in last.groupby(['regime', 'condition']):
        axes[0].scatter(block.effective_rank_a, block.cka, label=f'{regime}: {condition}')
        axes[1].plot(block.groupby('n_pairs').cka.mean(), 'o-', label=f'{regime}: {condition}')
    axes[0].set(xlabel='Effective covariance rank A', ylabel='CKA', title='Alignment and dimensionality')
    axes[1].set(xlabel='Training transitions', ylabel='Mean CKA', title='Descriptive means, not confidence bands')
    axes[0].legend(fontsize=7)
    fig.tight_layout()
    plt.show()
else:
    print('No results yet. The experiment checklist and configuration above define what to run.')

## Optional export and persistence

The results, training histories, exact plan, and environment can be packaged into ZIP bytes in RAM. No archive or automatic browser download is produced on your laptop. With `SAVE_RESULTS=False`, a runtime interruption loses progress; enable saving to an explicitly chosen persistent/cloud directory before a long run if desired.

The optional download link below works only when you choose to display it and click it. It is not invoked automatically. Checkpoints written in opt-in mode can resume matching runs; source, package versions, settings, and device contribute to identity. These are new runs, not silent reuse of legacy outputs.

In [ ]:
SHOW_DOWNLOAD_LINK = False
EXPORT_BYTES = bundle_zip(bundle) if bundle is not None else None
if EXPORT_BYTES is not None:
    print(f'{len(EXPORT_BYTES):,} export bytes prepared in RAM. Saving enabled: {SAVE_RESULTS}')
    if SHOW_DOWNLOAD_LINK:
        import base64
        from IPython.display import HTML
        display(HTML('<a download="alignment_collapse.zip" href="data:application/zip;base64,' +
                     base64.b64encode(EXPORT_BYTES).decode() + '">Click to download</a>'))
else:
    print('No completed training results to export. Nothing has been saved.')